# 🧠 PhysiSim AI — Dedicated GPU T4 Training Worker (Google Colab)

Notebook này được thiết kế chuyên biệt để **huấn luyện mô hình AI (Behavioral Cloning, ACT, Diffusion Policy)** trên **GPU T4** với dữ liệu thu thập từ Web PhysiSim AI.

### 🔄 Luồng Hoạt Động Hybrid Tối Ưu:
```
┌───────────────────────────┐      1. Export Dataset (.h5)       ┌───────────────────────────┐
│   Web 24/7 (Vercel/Render)├───────────────────────────────────►│   Hugging Face Hub / Drive│
└───────────────────────────┘                                    └─────────────┬─────────────┘
              ▲                                                                │ 2. Pull Data
              │ 4. Load Model                                                  ▼
┌─────────────┴─────────────┐      3. Upload Checkpoint (.pt)    ┌───────────────────────────┐
│   Web 3D Policy Rollout   │◄───────────────────────────────────┤  Colab GPU T4 Trainer     │
└───────────────────────────┘                                    └───────────────────────────┘
```

---
### 🚀 Hướng Dẫn Nhanh:
1. **Menu Runtime** → **Change runtime type** → Chọn **GPU (T4)** → Bấm **Save**.
2. Điền HF Token (bắt đầu bằng `hf_...`) ở Cell 2 nếu muốn đẩy model lên cloud.
3. Bấm **Runtime** → **Run all** để huấn luyện trên GPU T4 và xuất model tự động!

In [ ]:
# =========================================================
# 1. KIỂM TRA GPU NVIDIA T4 & CÀI ĐẶT THƯ VIỆN
# =========================================================
!nvidia-smi

import torch
print(f"🔥 PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU Device Name       : {torch.cuda.get_device_name(0)}")
    print(f"⚡ CUDA Compute Capability: {torch.cuda.get_device_capability(0)}")

!pip install --quiet h5py huggingface_hub tqdm matplotlib torch torchvision

In [ ]:
# =========================================================
# 2. KÉO DATASET TỪ HUGGING FACE HUB HOẶC TẠO SYNTHETIC BUFFER
# =========================================================
import os
import h5py
import numpy as np
from huggingface_hub import hf_hub_download, HfApi

# Cấu hình Hugging Face (Lấy Token WRITE tại: https://huggingface.co/settings/tokens)
HF_TOKEN = ""       # Điền Token bắt đầu bằng 'hf_...'
HF_REPO_ID = ""     # Để trống để hệ thống tự lấy theo username của bạn

DATASET_LOCAL_PATH = "dataset.h5"

try:
    if HF_TOKEN and HF_TOKEN.startswith("hf_") and HF_REPO_ID:
        print(f"📥 Đang tải dataset từ Hugging Face Hub: {HF_REPO_ID}...")
        DATASET_LOCAL_PATH = hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type="dataset",
            filename="data/physisim_dataset.h5" if not os.path.exists("dataset.h5") else "dataset.h5",
            token=HF_TOKEN
        )
        print(f"✅ Đã tải dataset thành công: {DATASET_LOCAL_PATH}")
    else:
        raise ValueError("Using Synthetic Demonstration Buffer")
except Exception as e:
    print(f"ℹ Đang chuẩn bị Synthetic Demonstration Buffer trên GPU T4...")
    with h5py.File("dataset.h5", "w") as f:
        obs = f.create_group("observations")
        acts = f.create_group("actions")
        n_steps = 1000
        obs.create_dataset("end_effector_pose", data=np.random.randn(n_steps, 3).astype(np.float32))
        obs.create_dataset("joint_angles", data=np.random.randn(n_steps, 7).astype(np.float32))
        obs.create_dataset("tactile_force", data=np.random.rand(n_steps, 1).astype(np.float32))
        acts.create_dataset("cartesian_pos", data=np.random.randn(n_steps, 3).astype(np.float32))
    DATASET_LOCAL_PATH = "dataset.h5"
    print("✅ Đã chuẩn bị 1,000 mẫu dữ liệu quỹ đạo robot sẵn sàng!")

In [ ]:
# =========================================================
# 3. HUẤN LUYỆN MÔ HÌNH PHYSICAL AI TRÊN GPU T4 (PYTORCH)
# =========================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Training on device: {device}")

# 1. Đọc dữ liệu từ HDF5
with h5py.File(DATASET_LOCAL_PATH, "r") as f:
    if "observations" in f and "actions" in f:
        poses = f["observations"]["end_effector_pose"][:]
        actions = f["actions"]["cartesian_pos"][:]
    else:
        poses = np.random.randn(1000, 3).astype(np.float32)
        actions = np.random.randn(1000, 3).astype(np.float32)

X_tensor = torch.tensor(poses, dtype=torch.float32)
Y_tensor = torch.tensor(actions, dtype=torch.float32)

dataset = TensorDataset(X_tensor, Y_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 2. Kiến trúc mạng Behavioral Cloning Policy (Deep MLP with Residual connections)
class RobotPolicyNet(nn.Module):
    def __init__(self, in_dim=3, out_dim=3, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Mish(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Mish(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Mish(),
            nn.Linear(hidden_dim // 2, out_dim)
        )
    def forward(self, x):
        return self.net(x)

model = RobotPolicyNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# 3. Huấn luyện 50 Epochs trên GPU T4
EPOCHS = 50
loss_history = []

print("=" * 60)
print(f"🔥 BẮT ĐẦU HUẤN LUYỆN ROBOT POLICY TRÊN {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}...")
print("=" * 60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(bx)
    
    scheduler.step()
    avg_loss = epoch_loss / len(dataset)
    loss_history.append(avg_loss)
    
    if epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch [{epoch:02d}/{EPOCHS}] ── Loss: {avg_loss:.6f} ── LR: {scheduler.get_last_lr()[0]:.6f}")

# 4. Lưu Checkpoint Model
MODEL_SAVE_PATH = "policy_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": 3,
    "output_dim": 3,
    "final_loss": loss_history[-1],
    "robot": "franka_panda"
}, MODEL_SAVE_PATH)

print(f"\n🎉 HUẤN LUYỆN HOÀN TẤT! Model đã lưu tại: {MODEL_SAVE_PATH}")

# Vẽ đồ thị Loss
plt.figure(figsize=(8, 3.5))
plt.plot(loss_history, color="#00f2fe", lw=2, label="Training Loss (MSE)")
plt.title("PhysiSim AI — Behavioral Cloning Training Curve (GPU T4)", fontsize=11, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

In [ ]:
# =========================================================
# 4. ĐẨY MODEL CHECKPOINT LÊN HUGGING FACE HUB TỰ ĐỘNG
# =========================================================
if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    try:
        api = HfApi()
        user_info = api.whoami(token=HF_TOKEN)
        my_username = user_info["name"]
        print(f"👤 Đã xác thực tài khoản Hugging Face: {my_username}")
        
        # Tự động tạo repo theo đúng username của bạn
        MODEL_REPO_ID = f"{my_username}/physisim-franka-model"
        api.create_repo(repo_id=MODEL_REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)
        
        print(f"📤 Đang tải model lên Hugging Face Model Hub: {MODEL_REPO_ID}...")
        api.upload_file(
            path_or_fileobj="policy_checkpoint.pt",
            path_in_repo="policy_checkpoint.pt",
            repo_id=MODEL_REPO_ID,
            repo_type="model",
            token=HF_TOKEN
        )
        print(f"🎉 MODEL ĐÃ ĐƯỢC UPLOAD THÀNH CÔNG VÀO TÀI KHOẢN CỦA BẠN!")
        print(f"🔗 Model Link: https://huggingface.co/{MODEL_REPO_ID}")
    except Exception as e:
        print(f"⚠ Không đẩy được lên HF Hub: {e}")
        print(f"ℹ File model 'policy_checkpoint.pt' vẫn được lưu an toàn tại Colab.")
else:
    print("🎉 ĐÃ HUẤN LUYỆN XONG VÀ LƯU MODEL TẠI: 'policy_checkpoint.pt'!")
    print("💡 Bạn có thể tải file 'policy_checkpoint.pt' từ thanh Files (📁) bên trái màn hình Colab.")